Première version : travail sur la qualité de l'air modélisation d'un seuil d'alerte sur la qualité de l'air. 


Module d'alerte complet : 


Bloc : API depuis base de données  fonction : récupérer la moyenne température qualité de l'air et humidité des 10 dernière min via la base de donnée :   appliqué  fiabilsiation concernant l'humidité puis appliquer l'indice atmos enfin création de l'alerte.

In [1]:
import time
from datetime import datetime
import pandas as pd
import smtplib 
import smtplib
from email.message import EmailMessage
import os 
from dotenv import load_dotenv
import sqlite3
import os
from dotenv import load_dotenv

# Charge les variables écrites dans le fichier .env
load_dotenv()

# --- Identifiants d'expédition (uniquement l'expéditeur dans le .env) ---
MAIL_SENDER = os.getenv("MAIL_SENDER")
MAIL_PASSWORD = os.getenv("MAIL_PASSWORD")
SMTP_SERVER = os.getenv("SMTP_SERVER", "smtp.gmail.com")
SMTP_PORT = int(os.getenv("SMTP_PORT", 465))

CHIP_ID = "14990683"
INTERVALLE_SCRAPING = 300  # 300 secondes = 5 minutes


def obtenir_dernieres_mesures(chip_id: str):
    date_du_jour = datetime.now().strftime("%Y-%m-%d")
    url = f"https://api-rrd.madavi.de/data_csv/csv-files/{date_du_jour}/data-esp8266-{chip_id}-{date_du_jour}.csv"

    try:
        # Téléchargement du CSV du jour
        df = pd.read_csv(url, sep=";")
        df["Time"] = pd.to_datetime(df["Time"])

        # Filtrage sur les 20 dernières minutes par rapport à la dernière écriture
        df["Time"] = pd.to_datetime(df["Time"]).dt.tz_localize("UTC").dt.tz_convert("Europe/Paris")
        derniere_date = df["Time"].max()
        seuil = derniere_date - pd.Timedelta(minutes=10)
        df_recent = df[df["Time"] >= seuil]

        # Calcul des moyennes
        pm25  = float(df_recent["SDS_P2"].mean())
        pm10 = float(df_recent["SDS_P1"].mean())
        hr = float(df_recent["BME280_humidity"].mean())  # Humidité relative moyenne
        temp =float(df_recent["BME280_temperature"].mean())# Température moyenne
        

        return round(pm25, 2), round(pm10, 2), round(hr, 2), round(temp, 2), derniere_date

    except Exception as e:
        print(f"[ERREUR] Échec de récupération : {e}")
        return None, None, None, None, None 


# Boucle de surveillance semi-temps réel
print("[DÉMARRAGE] Surveillance semi-temps réel via Madavi...")

    

[DÉMARRAGE] Surveillance semi-temps réel via Madavi...


Foction pour récupérer info base de donnée 

In [2]:
def obtenir_email_abonne(chip_id: str):
    conn = sqlite3.connect("air_quality_local.db")
    cursor = conn.cursor()
    cursor.execute("SELECT email FROM users WHERE numero_capteur = ? LIMIT 1;", (chip_id,))
    row = cursor.fetchone()
    conn.close()
    return row[0] if row else None

Fx d'envoie depuis depuis GMAIL 

In [3]:
def envoyer_alerte_depuis_base(destinataire: str, statut: str, niveau: int, pm25: float, pm10: float):
    msg = EmailMessage()
    msg["Subject"] = f"⚠️ Alerte Qualité de l'Air - Statut : {statut} (Niveau {niveau})"
    msg["From"] = MAIL_SENDER
    msg["To"] = destinataire

    msg.set_content(
        f"Bonjour,\n\n"
        f"Une dégradation de la qualité de l'air a été détectée sur votre capteur ({CHIP_ID}).\n\n"
        f"📊 Détails du relevé :\n"
        f"• Qualification ATMO : {statut} (Niveau {niveau}/6)\n"
        f"• PM 2.5 : {pm25:.1f} µg/m³\n"
        f"• PM 10  : {pm10:.1f} µg/m³\n\n"
        f"Pensez à limiter l'aération et à consulter votre tableau de bord."
    )

    try:
        with smtplib.SMTP_SSL(SMTP_SERVER, SMTP_PORT) as server:
            server.login(MAIL_SENDER, MAIL_PASSWORD)
            server.send_message(msg)
            print(f"✅ Alerte envoyée par email à {destinataire} !")
    except Exception as e:
        print(f"❌ Erreur lors de l'envoi de l'email : {e}")

Fx modifier qualitée de l'air en fonction de l'humidité 

In [4]:
def corriger_humidite_pm25(pm_valeur: float, HR: float, seuil_hr: float = 70.0, c: float = 0.25) -> float:
    """
    Corrige la surestimation des particules (PM2.5 / PM10) due à l'humidité relative.
    
    :param pm_valeur: Mesure brute du capteur (µg/m³)
    :param 
    HR
    : Humidité relative mesurée (%)
    :param seuil_hr: Seuil à partir duquel l'effet de l'eau devient mesurable (défaut: 70%)
    :param c: Facteur empirique hygroscopique (défaut: 0.25)
    :return: Valeur PM corrigée (µg/m³)
    """
    # Si l'humidité est normale, la mesure optique est fiable sans correction
    if HR <= seuil_hr:
        return round(pm_valeur, 2)
    
    # Sécurité pour éviter la division par zéro en cas de saturation extrême (condensation)
    HR_normalisee = min(HR, 98.0) / 100.0
    
    # Calcul du facteur de gonflement hygroscopique
    facteur_grossissement = 1.0 + c * (HR_normalisee ** 2) / (1.0 - HR_normalisee)
    
    pm_corrige = pm_valeur / facteur_grossissement
    return round(pm_corrige, 2)

fx modifier qualité de l'air en fonction del'humidité 


In [5]:
def corriger_humidite_pm_PM10(pm_valeur: float, HR: float, seuil_hr: float = 70.0, c: float = 0.18) -> float:
    """
    Corrige la surestimation des particules (PM2.5 / PM10) due à l'humidité relative.
    
    :param pm_valeur: Mesure brute du capteur (µg/m³)
    :param 
    HR
    : Humidité relative mesurée (%)
    :param seuil_hr: Seuil à partir duquel l'effet de l'eau devient mesurable (défaut: 70%)
    :param c: Facteur empirique hygroscopique (défaut: 0.25)
    :return: Valeur PM corrigée (µg/m³)
    """
    # Si l'humidité est normale, la mesure optique est fiable sans correction
    if HR <= seuil_hr:
        return round(pm_valeur, 2)
    
    # Sécurité pour éviter la division par zéro en cas de saturation extrême (condensation)
    HR_normalisee = min(HR, 98.0) / 100.0
    
    # Calcul du facteur de gonflement hygroscopique
    facteur_grossissement = 1.0 + c * (HR_normalisee ** 2) / (1.0 - HR_normalisee)
    
    pm_corrige = pm_valeur / facteur_grossissement
    return round(pm_corrige, 2)

bloc d'alert : 

In [6]:
# --- Seuils et Catégories ---
categories = [
    "Bon",
    "Moyen",
    "Dégradé",
    "Mauvais",
    "Très mauvais",
    "Extrêmement mauvais",
]

categories_temperature = [
    "Froid",
    "Bon",
    "Chaud",
    "Insoutenable",
    "Extrêmement chaud",
]
seuils_pm25 = [10, 20, 25, 50, 75]
seuils_pm10 = [20, 40, 50, 100, 150]
seuil_temperature = [0, 10, 20, 30, 40]

CHIP_ID = "14990683"
INTERVALLE_SCRAPING = 20  # secondes

print("[DÉMARRAGE] Surveillance semi-temps réel activée...")

# Récupération de l'email cible depuis la base SQLite
destinataire_email = obtenir_email_abonne(CHIP_ID)
if destinataire_email:
    print(f"📧 Destinataire configuré pour le capteur {CHIP_ID} : {destinataire_email}")
else:
    print(f"⚠️ Attention : aucun abonné trouvé en base pour le capteur {CHIP_ID} !")

alerte_deja_envoyee = False

while True:
    # 1. Récupération des données
    pm25, pm10, hr, temp, horodatage = obtenir_dernieres_mesures(CHIP_ID)

    if pm25 is not None and pm10 is not None and hr is not None and temp is not None:

        # 2. Correction hygrométrique si HR > 70%
        if hr > 70:
            print("Humidité relative > 70% : application de la correction de Hänel")
            pm25 = corriger_humidite_pm25(pm25, hr)
            pm10 = corriger_humidite_pm_PM10(pm10, hr)

        # 3. Qualification PM2.5
        pos_pm25 = len(seuils_pm25)
        for i in range(len(seuils_pm25)):
            if pm25 <= seuils_pm25[i]:
                pos_pm25 = i
                break
        niveau_pm25 = pos_pm25 + 1

        # 4. Qualification température
        pos_temperature = len(seuil_temperature)
        for i in range(len(seuil_temperature)):
            if temp <= seuil_temperature[i]:
                pos_temperature = i
                break
        niveau_temperature = pos_temperature + 1

        # 5. Qualification PM10
        pos_pm10 = len(seuils_pm10)
        for i in range(len(seuils_pm10)):
            if pm10 <= seuils_pm10[i]:
                pos_pm10 = i
                break
        niveau_pm10 = pos_pm10 + 1

        # 6. Règle ATMO : niveau le plus dégradé
        niveau_global = max(niveau_pm25, niveau_pm10)
        qualification_globale = categories[niveau_global - 1]

        # --- BLOC D'ALERTE PAR EMAIL ---
        if niveau_global > 1:
            print(f"Alerte : {qualification_globale} (Niveau {niveau_global})")
            
            if not alerte_deja_envoyee and destinataire_email:
                envoyer_alerte_depuis_base(
                    destinataire=destinataire_email, 
                    statut=qualification_globale,
                    niveau=niveau_global,
                    pm25=pm25,
                    pm10=pm10
                )
                alerte_deja_envoyee = True
        else:
            if alerte_deja_envoyee:
                print("✅ Retour à la normale : alerte réarmée.")
                alerte_deja_envoyee = False

        # 7. Logs
        print("--------------------------------------------------")
        print(f"[{horodatage}] Mesure (HR: {hr:.1f}%, Temp: {temp:.1f}°C) :")
        print(f" - PM2.5 : {pm25:.1f} µg/m³ (Niveau {niveau_pm25} : {categories[pos_pm25]})")
        print(f" - PM10  : {pm10:.1f} µg/m³ (Niveau {niveau_pm10} : {categories[pos_pm10]})")
        print(f" - Température : {temp:.1f} °C (Niveau {niveau_temperature} : {categories_temperature[pos_temperature]})")
        print(f" => Statut ATMO Global : {qualification_globale} (Niveau {niveau_global})")
        print("--------------------------------------------------")

    else:
        print(f"[{datetime.now()}] Capteur injoignable ou données incomplètes.")

    time.sleep(INTERVALLE_SCRAPING)

[DÉMARRAGE] Surveillance semi-temps réel activée...
📧 Destinataire configuré pour le capteur 14990683 : leo.panier55@gmail.com
Alerte : Moyen (Niveau 2)
✅ Alerte envoyée par email à leo.panier55@gmail.com !
--------------------------------------------------
[2026-11-13 21:53:26+01:00] Mesure (HR: 47.8%, Temp: 21.6°C) :
 - PM2.5 : 11.6 µg/m³ (Niveau 2 : Moyen)
 - PM10  : 17.8 µg/m³ (Niveau 1 : Bon)
 - Température : 21.6 °C (Niveau 4 : Insoutenable)
 => Statut ATMO Global : Moyen (Niveau 2)
--------------------------------------------------
Alerte : Moyen (Niveau 2)
--------------------------------------------------
[2026-11-13 21:55:51+01:00] Mesure (HR: 47.7%, Temp: 21.6°C) :
 - PM2.5 : 11.6 µg/m³ (Niveau 2 : Moyen)
 - PM10  : 18.3 µg/m³ (Niveau 1 : Bon)
 - Température : 21.6 °C (Niveau 4 : Insoutenable)
 => Statut ATMO Global : Moyen (Niveau 2)
--------------------------------------------------
Alerte : Moyen (Niveau 2)
--------------------------------------------------
[2026-11-13 21

KeyboardInterrupt: 